In [ ]:
import pandas as pd
df = pd.read_csv("data/creditcard.csv")
print(df.shape)
df.head()

In [ ]:
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True) * 100)
print(df.isnull().sum().sum(), df.duplicated().sum())
df[["Time", "Amount"]].describe()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape, df["Class"].value_counts())

# 2. Features and target
X = df.drop("Class", axis=1)
y = df["Class"]

# 3. Stratified split BEFORE any scaling or resampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. Scale Amount and Time: fit on train only
scaler = RobustScaler()
X_train[["Amount", "Time"]] = scaler.fit_transform(X_train[["Amount", "Time"]])
X_test[["Amount", "Time"]] = scaler.transform(X_test[["Amount", "Time"]])

print(y_train.mean(), y_test.mean())  # both should be about 0.0017

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score)

def evaluate(name, model, X_te, y_te):
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"\n=== {name} ===")
    print(confusion_matrix(y_te, pred))
    print(classification_report(y_te, pred, digits=4))
    print("PR-AUC :", round(average_precision_score(y_te, proba), 4))
    print("ROC-AUC:", round(roc_auc_score(y_te, proba), 4))

# Model 1: Logistic Regression with class weights
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
evaluate("Logistic Regression", lr, X_test, y_test)

# Model 2: XGBoost with imbalance weighting
spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                    scale_pos_weight=spw, eval_metric="aucpr",
                    random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
evaluate("XGBoost", xgb, X_test, y_test)

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. SMOTE + XGBoost, with SMOTE inside the pipeline so CV doesn't leak
smote_pipe = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                          eval_metric="aucpr", random_state=42, n_jobs=-1))
])
smote_pipe.fit(X_train, y_train)
evaluate("XGBoost + SMOTE", smote_pipe, X_test, y_test)

# 2. Cross-validated PR-AUC for your class-weighted XGBoost
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(xgb, X_train, y_train, cv=cv,
                         scoring="average_precision", n_jobs=-1)
print("CV PR-AUC:", scores.mean().round(4), "+/-", scores.std().round(4))

In [ ]:
import joblib, os
os.makedirs("models", exist_ok=True)

joblib.dump(xgb, "models/xgb_model.joblib")
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(list(X_train.columns), "models/feature_columns.joblib")
joblib.dump(float(best), "models/threshold.joblib")
print(os.listdir("models"))

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba = cross_val_predict(xgb, X_train, y_train, cv=cv,
                              method="predict_proba", n_jobs=-1)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best = float(thr[np.argmax(f1)])
print("Best threshold (from CV on train):", round(best, 3), "| CV F1:", round(f1.max(), 4))

# Now check it once on the untouched test set
test_proba = xgb.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best).astype(int)
print("Test precision:", round(precision_score(y_test, test_pred), 4))
print("Test recall   :", round(recall_score(y_test, test_pred), 4))
print("Test F1       :", round(f1_score(y_test, test_pred), 4))

In [ ]:
import joblib, os
os.makedirs("models", exist_ok=True)

joblib.dump(xgb, "models/xgb_model.joblib")
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(list(X_train.columns), "models/feature_columns.joblib")
joblib.dump(best, "models/threshold.joblib")
print(os.listdir("models"))

In [ ]:
import json
sample = df[df["Class"] == 1].iloc[0].drop("Class").to_dict()
print(json.dumps(sample, indent=2))

In [ ]:
row = df[df["Class"] == 1].iloc[0]
print(row[["V24", "V25", "V26"]].to_dict())

In [ ]:
import pandas as pd, json
df = pd.read_csv("data/creditcard.csv")
row = df[df.Class == 1].iloc[0].drop("Class")
print(json.dumps(row.to_dict()))

In [ ]:
import pandas as pd, requests
df = pd.read_csv("data/creditcard.csv")
row = df[df.Class == 1].iloc[0].drop("Class")
r = requests.post("http://127.0.0.1:8000/predict", json=row.to_dict())
print(r.status_code)
print(r.json())

In [ ]:
row = df[df.Class == 0].iloc[0].drop("Class")
r = requests.post("http://127.0.0.1:8000/predict", json=row.to_dict())
print(r.json())